# Flag Additions Optimize

Optimizes file removing escape characters and decreasing the flag size

In [15]:
import requests
import json
import urllib
import re

## Open the file

In [16]:
flagsPath = "./out/flags.json"

with open(flagsPath, "r", encoding="utf-8") as json_file:
    flags = json.load(json_file)

## Unescape characters in flags links

In [17]:

for country in flags:
    print(country)
    country = flags[country]
    if(not 'regions' in country): continue
    regions = country['regions']
    for region in regions.values():
        if(not 'flag' in region): continue
        flag = region['flag']
        region['flag'] = urllib.parse.unquote(flag)


AL
AD
AR
AU
AT
BH
BY
BE
BO
BA
BR
CA
CL
CN
CO
CR
HR
CZ
DK
EC
EG
SV
EE
FI
FR
GE
DE
GT
HN
HU
ID
IQ
IE
IT
JP
KR
KW
KG
LV
LT
MY
MX
MN
MM
NL
NZ
NI
NO
PK
PY
PE
PH
PL
PT
RO
RU
SK
SO
ZA
ES
LK
SE
CH
TH
UA
AE
GB
US
UY
UZ
VE
AF
AM
AO
BB
BD
BF
BG
BN
DO
DZ
GR
IL
IN
IS
JM
JO
KH
KZ
LA
LB
LU
MA
MK
NP
OM
QA
RS
SA
SG
SI
TN
TR
TT
TW
VN


## Reduce the flag size

### Undo previous png optimization

In [18]:
for country in flags.values():
    if(not 'regions' in country): continue
    for flagData in country["regions"].values():
        if(not 'flag' in flagData): continue
        flag: str = flagData["flag"]
        if "upload.wikimedia.org" not in flag:
            continue

        if "/thumb/" in flag:
            flagData["flag"] = re.sub(r"/(\d+px-)", "/120px-", flag, count=1)
            print(flagData["flag"])
            continue

        textToMatch = ".svg"
        containsSvg = flag.find(textToMatch)
        if containsSvg != -1:
            flagData["flag"] = flag[:containsSvg + len(textToMatch)].replace("/thumb", "")
            print(flagData["flag"])

https://upload.wikimedia.org/wikipedia/commons/4/49/Flag_of_Berat.svg
https://upload.wikimedia.org/wikipedia/commons/e/e4/Flag_of_Durrës.svg
https://upload.wikimedia.org/wikipedia/commons/4/4d/Flag_of_Elbasan.svg
https://upload.wikimedia.org/wikipedia/commons/2/2b/Flag_of_Korçë.svg
https://upload.wikimedia.org/wikipedia/commons/7/7e/Flag_of_Kukës.svg
https://upload.wikimedia.org/wikipedia/commons/1/13/Flag_of_Tiranë.svg
https://upload.wikimedia.org/wikipedia/commons/3/30/Flag_of_Vlorë.svg
https://upload.wikimedia.org/wikipedia/commons/2/29/Stema_e_Qarkut_Fier.svg
https://upload.wikimedia.org/wikipedia/commons/4/4e/Stema_e_Qarkut_Gjirokastër.svg
https://upload.wikimedia.org/wikipedia/commons/b/b3/Stema_e_Qarkut_Lezhë.svg
https://upload.wikimedia.org/wikipedia/commons/4/49/Stema_e_Qarkut_Shkodër.svg
https://upload.wikimedia.org/wikipedia/commons/6/6a/Flag_of_Andorra_la_Vella.svg
https://upload.wikimedia.org/wikipedia/commons/8/87/Flag_of_Canillo.svg
https://upload.wikimedia.org/wikipedia

In [19]:
image_url = "https://upload.wikimedia.org/wikipedia/commons/4/43/Flag_of_Southwest_Papua.svg"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36"}


def urlContentSize(url):
    response = requests.get(url, headers=headers)

    if response.status_code == 200:
        file_size_bytes = len(response.content)
        return file_size_bytes / 1024
    else:
        print(
            f"Failed to fetch the image {url} . Status code: {response.status_code}")

def request_with_retry(url, timeout=30, params=None, allow_redirects=True, sleep_before=2.0, max_retries=5):
    if not url:
        raise ValueError("URL is required")

    for attempt in range(max_retries):
        if attempt > 0:
            print(f"Retrying request for {url} ({attempt + 1}/{max_retries})")

        if sleep_before:
            time.sleep(sleep_before)

        try:
            response = requests.get(
                url,
                params=params,
                headers=headers,
                allow_redirects=allow_redirects,
                timeout=timeout,
            )
            if response.status_code == 429:
                if attempt < max_retries - 1:
                    print(f"Rate limited (429) for {url}. Waiting 60 seconds before retrying...")
                    time.sleep(60)
                    continue
                response.raise_for_status()
            response.raise_for_status()
            return response
        except requests.RequestException as exc:
            if getattr(exc.response, "status_code", None) == 429 and attempt < max_retries - 1:
                print(f"Rate limited (429) for {url}. Waiting 60 seconds before retrying...")
                time.sleep(60)
                continue
            raise

    raise RuntimeError(f"Failed to fetch {url} after {max_retries} attempts")

In [20]:
import time
import urllib.parse
import re
import requests

checked_urls = {}

with open(flagsPath, "r", encoding="utf-8") as json_file:
    flags = json.load(json_file)


def wikimedia_title_from_url(url: str) -> str:
    path = urllib.parse.unquote(urllib.parse.urlparse(url).path)
    filename = path.rsplit("/", 1)[-1]
    if "/thumb/" in path:
        filename = re.sub(r"^\d+px-", "", filename)

    if re.search(r"\.(svg|png|jpg|jpeg|gif|webp)\.(png|jpg|jpeg|gif|webp)$", filename, re.I):
        filename = re.sub(r"\.(png|jpg|jpeg|gif|webp)$", "", filename, flags=re.I)

    filename = filename.replace("_", " ")
    return f"File:{filename}"





def url_exists(url, timeout=8):
    if not url:
        return False

    if url in checked_urls:
        return checked_urls[url]

    try:
        response = request_with_retry(url, timeout=timeout, allow_redirects=True)
        status_ok = response.status_code in (200, 206)
        ctype = (response.headers.get("content-type") or "").lower()
        path_lower = urllib.parse.urlparse(url).path.lower()
        ext = path_lower.rsplit("/", 1)[-1].rsplit(".", 1)[-1] if "." in path_lower.rsplit("/", 1)[-1] else ""
        has_image_ext = ext.lower() in {"png", "jpg", "jpeg", "gif", "svg", "webp"}
        content = response.content if hasattr(response, "content") else b""

        if status_ok and (ctype.startswith("image/") or ctype in {"application/octet-stream", "application/xml", "text/xml"}):
            result = True
        elif status_ok and has_image_ext and len(content) > 0:
            result = True
        elif status_ok and ctype.startswith("text/") and ext.lower() == "svg":
            result = b"<svg" in content.lower()
        else:
            result = False

        if not result and status_ok and has_image_ext and ctype.startswith("text/"):
            result = b"<svg" in content.lower()
    except Exception:
        result = False

    checked_urls[url] = result
    return result


def normalize_wikimedia_thumb_url(url, width=120):
    if "upload.wikimedia.org" not in url:
        return url

    if "/thumb/" in url:
        return re.sub(r"/(\d+px-)", f"/{width}px-", url, count=1)

    match = re.match(
        r"^https://upload\.wikimedia\.org/wikipedia/commons/([^/]+)/([^/]+)/(.+?)(?:\.(png|jpg|jpeg|svg|gif))$",
        url,
    )
    if match:
        prefix = f"https://upload.wikimedia.org/wikipedia/commons/thumb/{match.group(1)}/{match.group(2)}/{match.group(3)}.{match.group(4)}/"
        return prefix + f"{width}px-{match.group(3)}.{match.group(4)}"

    return url


def build_wikimedia_fallback_urls(url):
    candidates = []
    if "/thumb/" not in url:
        return candidates

    match = re.match(
        r"^https://upload\.wikimedia\.org/wikipedia/commons/thumb/([^/]+)/([^/]+)/(.+?)/(\d+px-.+)$",
        url,
    )
    if not match:
        return candidates

    prefix = f"https://upload.wikimedia.org/wikipedia/commons/{match.group(1)}/{match.group(2)}/"
    filename = match.group(3)
    candidates.append(prefix + filename)

    if "." in filename:
        stem, ext = filename.rsplit(".", 1)
        if ext.lower() != "svg":
            candidates.append(prefix + stem + ".svg")

    return candidates


def check_wikimedia_size_bulk(flag_urls, batch_size=50):
    results = {}
    api_url = "https://commons.wikimedia.org/w/api.php"

    unique_urls = list(dict.fromkeys(flag_urls))

    for start in range(0, len(unique_urls), batch_size):
        batch = unique_urls[start:start + batch_size]
        titles = [wikimedia_title_from_url(url) for url in batch]
        params = {
            "action": "query",
            "format": "json",
            "titles": "|".join(titles),
            "prop": "imageinfo",
            "iiprop": "size|url|thumburl",
            "iiurlwidth": 120,
            "formatversion": 2,
        }

        try:
            response = request_with_retry(api_url, timeout=30, params=params)
            payload = response.json()
            pages = payload.get("query", {}).get("pages", [])
            normalized = payload.get("query", {}).get("normalized", [])
            title_lookup = {item["from"]: item["to"] for item in normalized}

            page_by_title = {}
            for page in pages:
                title = page.get("title")
                if title:
                    page_by_title[title] = page

            for url, title in zip(batch, titles):
                canonical_title = title_lookup.get(title, title)
                page = page_by_title.get(canonical_title) or page_by_title.get(title)
                if not page:
                    page = {}
                imageinfo = (page.get("imageinfo") or [{}])[0]
                results[url] = {
                    "size_bytes": imageinfo.get("size"),
                    "thumb_url": imageinfo.get("thumburl"),
                    "original_url": imageinfo.get("url"),
                    "exists": not page.get("missing", False),
                }
        except Exception as exc:
            print(f"API request failed: {exc}")
            for url in batch:
                results[url] = {"size_bytes": None, "thumb_url": None, "original_url": None, "exists": False}

        if start + batch_size < len(unique_urls):
            time.sleep(10)

    return results


flag_urls = []
broken_links = []
for country in flags.values():
    if "regions" not in country:
        continue
    for flagData in country["regions"].values():
        if "flag" not in flagData:
            continue
        flag: str = flagData["flag"]
        if "wikimedia.org" in flag:
            flag_urls.append(flag)

size_results = check_wikimedia_size_bulk(flag_urls)

for country in flags.values():
    if "regions" not in country:
        continue
    for flagData in country["regions"].values():
        if "flag" not in flagData:
            continue
        flag: str = flagData["flag"]
        if "wikimedia.org" not in flag:
            continue

        info = size_results.get(flag, {})
        size_bytes = info.get("size_bytes")
        candidate_urls = []

        if info.get("exists"):
            if size_bytes is not None and size_bytes > 100 * 1024 and info.get("thumb_url"):
                candidate_urls.append(info["thumb_url"])
            if info.get("original_url"):
                candidate_urls.append(info["original_url"])

        candidate_urls.append(flag)
        candidate_urls.append(normalize_wikimedia_thumb_url(flag))
        candidate_urls.extend(build_wikimedia_fallback_urls(flag))
        candidate_urls = list(dict.fromkeys([candidate for candidate in candidate_urls if candidate]))

        resolved_url = None
        if info.get("exists"):
            if size_bytes is not None and size_bytes > 100 * 1024 and info.get("thumb_url"):
                resolved_url = info["thumb_url"]
            elif info.get("original_url"):
                resolved_url = info["original_url"]
            elif info.get("thumb_url"):
                resolved_url = info["thumb_url"]
        else:
            for candidate in candidate_urls:
                if candidate and url_exists(candidate):
                    resolved_url = candidate
                    break

        if resolved_url:
            flagData["flag"] = resolved_url
            print(f"Resolved link for {flag} -> {resolved_url}")
        else:
            broken_links.append({"original": flag, "candidates": candidate_urls})
            flagData["flag"] = flag
            print(f"BROKEN LINK (manual review needed): {flag}")

if broken_links:
    print(f"\nLinks needing manual review: {len(broken_links)}")
    for item in broken_links[:20]:
        print(item["original"])
        for candidate in item["candidates"]:
            print("  -", candidate)


Resolved link for https://upload.wikimedia.org/wikipedia/commons/4/49/Flag_of_Berat.svg -> https://upload.wikimedia.org/wikipedia/commons/4/49/Flag_of_Berat.svg
Resolved link for https://upload.wikimedia.org/wikipedia/commons/e/e4/Flag_of_Durrës.svg -> https://upload.wikimedia.org/wikipedia/commons/e/e4/Flag_of_Durr%C3%ABs.svg
Resolved link for https://upload.wikimedia.org/wikipedia/commons/4/4d/Flag_of_Elbasan.svg -> https://upload.wikimedia.org/wikipedia/commons/4/4d/Flag_of_Elbasan.svg
Resolved link for https://upload.wikimedia.org/wikipedia/commons/2/2b/Flag_of_Korçë.svg -> https://upload.wikimedia.org/wikipedia/commons/2/2b/Flag_of_Kor%C3%A7%C3%AB.svg
Resolved link for https://upload.wikimedia.org/wikipedia/commons/7/7e/Flag_of_Kukës.svg -> https://upload.wikimedia.org/wikipedia/commons/7/7e/Flag_of_Kuk%C3%ABs.svg
Resolved link for https://upload.wikimedia.org/wikipedia/commons/1/13/Flag_of_Tiranë.svg -> https://upload.wikimedia.org/wikipedia/commons/1/13/Flag_of_Tiran%C3%AB.svg
R

In [ ]:

for country in flags:
    print(country)
    country = flags[country]
    if(not 'regions' in country): continue
    regions = country['regions']
    for region in regions.values():
        if(not 'flag' in region): continue
        flag = region['flag']
        region['flag'] = urllib.parse.unquote(flag)


In [21]:
json_data = json.dumps(
    flags, indent=4, ensure_ascii=False).encode('utf8').decode()

with open(flagsPath, "w") as json_file:
    json_file.write(json_data)

print("JSON data has been saved to", flagsPath)


JSON data has been saved to ./out/flags.json
